In [10]:
import pandas as pd
import json

with open('data.json') as f:
    records = json.load(f)

df = pd.json_normalize(records)
df = df.sort_values('data.timestamp').reset_index(drop=True)

# 1. Expand the 'data.data' list column into separate columns globally
unpacked_values = pd.DataFrame(df['data.data'].tolist(), index=df.index)

# 2. Add standard/fallback column names to the unpacked DataFrame
max_cols = unpacked_values.shape[1]
unpacked_values.columns = [f'value_{i}' for i in range(max_cols)]

# 3. Combine the metadata and the unpacked values side-by-side
df = pd.concat([df, unpacked_values], axis=1)

# 4. Map sensor-specific names to the unpacked columns dynamically
SENSOR_COLUMNS = {
    'Orientation Sensor': ['azimuth', 'pitch', 'roll'],
    'Rotation Vector Sensor': ['rot_x', 'rot_y', 'rot_z', 'rot_w', 'heading_accuracy'],
    'Goldfish Orientation sensor': ['azimuth', 'pitch', 'roll'],
    'Goldfish Ambient Temperature sensor': ['temperature'],
    'Gravity Sensor': ['gravity_x', 'gravity_y', 'gravity_z'],
}

for sensor_id, target_cols in SENSOR_COLUMNS.items():
    # Identify rows belonging to this sensor
    mask = df['data.sensorId'] == sensor_id
    if mask.any():
        # Map 'value_0', 'value_1', etc. to their named counterparts
        rename_dict = {f'value_{i}': col for i in range(len(target_cols)) for col in [target_cols[i]]}
        
        # We use .loc to update the specific values for these rows safely
        for val_col, new_name in rename_dict.items():
            if val_col in df.columns:
                if new_name not in df.columns:
                    df[new_name] = pd.NA  # Initialize column if it doesn't exist
                df.loc[mask, new_name] = df.loc[mask, val_col]

# 5. Relative time per sensor stream
df['relative_time_sec'] = (
    df['data.timestamp'] - df.groupby('data.sensorId')['data.timestamp'].transform('min')
) / 1e9

# 6. Extract target DataFrames safely
orientation_df = df[df['data.sensorId'] == 'Orientation Sensor'][
    ['relative_time_sec', 'azimuth', 'pitch', 'roll']
].dropna(subset=['azimuth'])

rotation_df = df[df['data.sensorId'] == 'Rotation Vector Sensor'][
    ['relative_time_sec', 'rot_x', 'rot_y', 'rot_z', 'rot_w', 'heading_accuracy']
].dropna(subset=['rot_x'])

print(orientation_df.head())

    relative_time_sec    azimuth     pitch      roll
0                0.00       -0.0 -85.25003       0.0
5                0.05   359.9985 -85.25209  0.000879
6                0.10  359.99673 -85.25378  0.001657
9                0.15  359.99402 -85.25515  0.002918
14               0.20  359.99246 -85.25645  0.003325


In [12]:
import matplotlib.pyplot as plt

# --- 1. VISUALIZE ORIENTATION SENSOR DATA ---
# Create a 3-row stacked subplot sharing the same time axis
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(10, 7))

# Plot Azimuth
axes[0].plot(orientation_df['relative_time_sec'], orientation_df['azimuth'], color='#d62728', linewidth=1.5, label='Azimuth')
axes[0].set_ylabel('Azimuth (°)')
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend(loc='upper right')

# Plot Pitch
axes[1].plot(orientation_df['relative_time_sec'], orientation_df['pitch'], color='#1f77b4', linewidth=1.5, label='Pitch')
axes[1].set_ylabel('Pitch (°)')
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend(loc='upper right')

# Plot Roll
axes[2].plot(orientation_df['relative_time_sec'], orientation_df['roll'], color='#ff7f0e', linewidth=1.5, label='Roll')
axes[2].set_ylabel('Roll (°)')
axes[2].set_xlabel('Relative Time (seconds)')
axes[2].grid(True, linestyle='--', alpha=0.5)
axes[2].legend(loc='upper right')

plt.suptitle('Orientation Sensor Dynamics Over Time', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('orientation_sensor_plot.png', dpi=300)
plt.close()


# --- 2. VISUALIZE ROTATION VECTOR SENSOR DATA ---
# Create a 2-row subplot: one for unit quaternion components, one for accuracy
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(10, 6))

# Plot Quaternion components (rot_x, rot_y, rot_z, rot_w) together
axes[0].plot(rotation_df['relative_time_sec'], rotation_df['rot_x'], label='rot_x', alpha=0.8)
axes[0].plot(rotation_df['relative_time_sec'], rotation_df['rot_y'], label='rot_y', alpha=0.8)
axes[0].plot(rotation_df['relative_time_sec'], rotation_df['rot_z'], label='rot_z', alpha=0.8)
axes[0].plot(rotation_df['relative_time_sec'], rotation_df['rot_w'], label='rot_w', alpha=0.8)
axes[0].set_ylabel('Quaternion Value')
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend(loc='upper right', ncol=4) # Clean horizontal legend placement

# Plot Heading Accuracy
axes[1].plot(rotation_df['relative_time_sec'], rotation_df['heading_accuracy'], color='#9467bd', linewidth=1.2, label='Heading Accuracy')
axes[1].set_ylabel('Accuracy (°)')
axes[1].set_xlabel('Relative Time (seconds)')
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend(loc='upper right')

plt.suptitle('Rotation Vector (Quaternion) & Accuracy Over Time', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('rotation_vector_plot.png', dpi=300)
plt.close()